# Self-Evolving Digital Twin: LSTM RUL + Drift-Aware Adaptive Learning (Google Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harishbala123/self-evolving-digital-twin/blob/main/notebooks/colab_training.ipynb)

NASA C-MAPSS FD001 LSTM Remaining-Useful-Life model, concept-drift detection (Page-Hinkley / ADWIN), drift-triggered retraining and a fleet digital-twin replay.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
The notebook clones the repository and runs the same scripts used on Windows, so the code is identical in both environments.
Colab already ships TensorFlow/Keras, so only the missing packages are installed. This avoids replacing Colab's CUDA-matched TensorFlow build.

In [ ]:
!nvidia-smi -L || echo 'No GPU: select Runtime > Change runtime type > T4 GPU'

In [ ]:
import os
if not os.path.exists('/content/self-evolving-digital-twin'):
    !git clone -q https://github.com/Harishbala123/self-evolving-digital-twin.git /content/self-evolving-digital-twin
%cd /content/self-evolving-digital-twin
!git log -1 --oneline

In [ ]:
!pip install -q pyyaml
import tensorflow as tf, keras; print('TF', tf.__version__, 'Keras', keras.__version__, tf.config.list_physical_devices('GPU'))

## 1. Acquire and validate the dataset

In [ ]:
!python scripts/download_data.py

## 2. Preprocess + train the LSTM (cuDNN-accelerated on GPU)

In [ ]:
!python scripts/train.py

## 3. Evaluate (train / validation / test RMSE, MAE, R², NASA score, graphs)

In [ ]:
!python scripts/evaluate.py
!cat results/metrics/evaluation_metrics.csv

## 4. Drift detection + retraining experiments

In [ ]:
!python scripts/run_drift_experiment.py
!cat results/metrics/retraining_before_after.csv

## 5. Fleet digital-twin demonstration

In [ ]:
!python scripts/run_digital_twin_demo.py
!cat results/metrics/twin_performance_by_phase_version.csv

## 6. Graphs

In [ ]:
from IPython.display import Image, display
display(Image('results/graphs/loss_curves.png'))
display(Image('results/graphs/test_pred_vs_actual.png'))
display(Image('results/graphs/drift_score_example.png'))
display(Image('results/graphs/retraining_before_after.png'))
display(Image('results/graphs/twin_fleet_timeline.png'))

## 7. Unit tests

In [ ]:
!pip install -q pytest && python -m pytest -q

## Export artifacts
The zip holds the trained model(s), metrics and graphs. Unzip it into the local project folder (Windows) to use
the Colab-trained model with the local evaluation and demo scripts. Mounting Google Drive is optional and only
needed if you want a persistent copy; no credentials are stored in the notebook.

In [ ]:
!zip -qr /content/self-evolving-digital-twin_artifacts.zip models results && ls -lh /content/self-evolving-digital-twin_artifacts.zip

In [ ]:
SAVE_TO_DRIVE = False   # set True to copy the zip to Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp /content/self-evolving-digital-twin_artifacts.zip /content/drive/MyDrive/
else:
    from google.colab import files
    files.download('/content/self-evolving-digital-twin_artifacts.zip')